# Baseline 0.6714 复现（JupyterLab）

> 详细说明见 [`baseline_0.6714.md`](../baseline_0.6714.md) **版本 B**  
> 内核请选择：**CCKS RePS (0.6714)**（`easyedit_reps/.venv`）

**使用前修改下一格中的 `PROJECT_ROOT` 和 `REPS_MODEL_PATH`。**

In [ ]:
from pathlib import Path
import os
import sys

# ===== 改成你的路径 =====
PROJECT_ROOT = Path("/path/to/CCKS-Steering-RePS-Structure").resolve()
REPS_MODEL_PATH = "/path/to/Qwen3-4B-Instruct-2507"

assert (PROJECT_ROOT / "baseline" / "submission.json").exists(), "请修改 PROJECT_ROOT"
os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["REPS_MODEL_PATH"] = REPS_MODEL_PATH

EE = PROJECT_ROOT / "easyedit_reps"
ED = EE / "EasyEdit"
VENV_SITE = EE / ".venv" / f"lib/python{sys.version_info.major}.{sys.version_info.minor}/site-packages"
CONDA_SITE = os.environ.get("CONDA_SITE", "")

for p in [str(VENV_SITE), str(ED), CONDA_SITE]:
    if p and Path(p).exists() and p not in sys.path:
        sys.path.insert(0, p)

os.environ["EASYEDIT_REPS_ROOT"] = str(EE)
os.environ["EASYEDIT_ROOT"] = str(ED)
os.chdir(EE)

import torch
import transformers
print("PROJECT_ROOT", PROJECT_ROOT)
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
print("cwd", os.getcwd())

In [ ]:
import json

layers = json.loads((PROJECT_ROOT / "baseline/layers.json").read_text(encoding="utf-8"))
mult = json.loads((PROJECT_ROOT / "baseline/multipliers.json").read_text(encoding="utf-8"))
print("L2_2 → layer", layers["concepts"]["L2_2"]["layer"], "mult", mult["L2_2"])

v = PROJECT_ROOT / "easyedit_reps/outputs/vectors/per_layer/L2_2/layer_18/steer_eval_concept_L2_2/reps_vector/layer_18.pt"
assert v.exists(), f"缺少向量，见复现指南 §3.4: {v}"
print("L2_2 vector OK", v.stat().st_size // 1024, "KB")

In [ ]:
import subprocess

# 全量重生成 ~1–2h，建议在 screen 中跑
cmd = ["bash", str(PROJECT_ROOT / "scripts/regen_from_baseline.sh"), "512", "jupyter_run"]
proc = subprocess.run(cmd, cwd=str(PROJECT_ROOT), env=os.environ.copy(), capture_output=True, text=True)
print(proc.stdout)
if proc.returncode:
    print(proc.stderr)
    raise RuntimeError("regen failed")

out = PROJECT_ROOT / "绝地邮兵_result_regen_jupyter_run.json"
print("written", out)

In [ ]:
import json

ref = json.load(open(PROJECT_ROOT / "baseline/submission.json", encoding="utf-8"))
new = json.load(open(PROJECT_ROOT / "绝地邮兵_result_regen_jupyter_run.json", encoding="utf-8"))

diff = 0
for br, bn in zip(ref, new):
    for gr, gn in zip(br["generated_results"], bn["generated_results"]):
        if gr["pred"] != gn["pred"]:
            diff += 1
            print("DIFF", br["concept_id"], gr["input"][:70])
print("diff samples", diff, "/ 120")